# Phase 2 Lab — Reference Solution

**Phase:** NumPy and Pandas  
**Scenario:** A monthly retail extract must be converted from messy transactions into a trusted analytical table.

**Deliverable:** A reproducible cleaning and aggregation pipeline plus a machine-readable quality report.

Use this only after completing your own attempt. Compare design decisions—not merely output.

## Requirements

        1. Declare expected columns, types, value ranges, and key uniqueness.
2. Parse dates with explicit invalid handling.
3. Standardize category and region text.
4. Resolve duplicates under a documented key.
5. Treat missing and impossible numeric values using a justified rule.
6. Create monthly/category summaries and validate totals.
7. Export cleaned data and a quality report.

        ## Acceptance criteria

        - The notebook runs from a clean kernel in order.
        - Inputs and outputs have explicit contracts.
        - Invalid, missing, extreme, duplicate, and unseen cases are considered.
        - Important invariants use assertions or tests.
        - Results include interpretation and limitations.
        - Generated artifacts are written under the course `artifacts/` folder.

## Planning worksheet

Complete before coding:

| Question | Your answer |
|---|---|
| What decision or user does the result serve? | |
| What does one row/object/event represent? | |
| What are the required inputs and types? | |
| What outputs and side effects are allowed? | |
| Which assumptions are most fragile? | |
| What is the simplest valid baseline? | |
| Which edge cases must be tested? | |
| How will you know the result is correct? | |

In [ ]:
from pathlib import Path
import sys
import json
import warnings
warnings.filterwarnings("ignore")

_candidates = [Path.cwd(), *Path.cwd().parents]
COURSE_ROOT = next((p for p in _candidates if (p / "datasets").exists()), Path.cwd())
DATA_DIR = COURSE_ROOT / "datasets"
ARTIFACT_DIR = COURSE_ROOT / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)
sys.path.insert(0, str(COURSE_ROOT))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print(f"Course root: {COURSE_ROOT}")

## Reference implementation

This is one defensible solution, not the only correct design. Identify at least one improvement before adopting it.

In [ ]:
from src.course_utils import dataframe_audit

raw = pd.read_csv(DATA_DIR/"messy_retail_sales.csv")
expected = {"order_id","order_date","region","category","units","unit_price","discount_pct","revenue","cost","profit"}
assert expected.issubset(raw.columns)

quality_before = dataframe_audit(raw)
clean = raw.drop_duplicates(subset=["order_id"], keep="first").copy()
clean["order_date"] = pd.to_datetime(clean["order_date"], errors="coerce")
clean["category"] = clean["category"].astype("string").str.strip().str.title()
clean["region"] = clean["region"].astype("string").str.strip().str.title().fillna("Unknown")
clean.loc[clean["units"] <= 0, "units"] = np.nan
clean.loc[~clean["discount_pct"].between(0,1), "discount_pct"] = np.nan
clean["unit_price"] = clean.groupby("category")["unit_price"].transform(lambda s: s.fillna(s.median()))
clean["units"] = clean["units"].fillna(clean["units"].median())
clean["discount_pct"] = clean["discount_pct"].fillna(clean["discount_pct"].median())
clean = clean.dropna(subset=["order_date"])
clean["month"] = clean["order_date"].dt.to_period("M").astype(str)

monthly = clean.groupby(["month","category"],as_index=False).agg(
    orders=("order_id","nunique"),
    units=("units","sum"),
    revenue=("revenue","sum"),
    profit=("profit","sum"),
)
assert clean["order_id"].is_unique
assert clean["discount_pct"].between(0,1).all()
assert (clean["units"] > 0).all()

display(quality_before)
display(monthly.head(12))
clean.to_csv(ARTIFACT_DIR/"phase2_clean_retail.csv",index=False)
monthly.to_csv(ARTIFACT_DIR/"phase2_monthly_summary.csv",index=False)
print("Outputs written to", ARTIFACT_DIR)

## Solution review

Review the reference under four lenses:

1. **Correctness:** Are contracts and calculations enforced?
2. **Robustness:** What failures remain unhandled?
3. **Maintainability:** Which responsibilities should become modules/functions?
4. **Decision validity:** Do outputs support the stated use without overclaiming?

Extend the implementation with one additional test and one observability improvement.